# QLoRA Fine-Tuning for Multimodal Alignment of Voxtral with GLaDOS Persona

In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoProcessor, VoxtralForConditionalGeneration, BitsAndBytesConfig
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
import os, gc, json, wandb, warnings

warnings.filterwarnings("ignore", category=UserWarning, module="bitsandbytes")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["WANDB_PROJECT"] = "Voxtral-GLaDOS-Multimodal"
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_NOTEBOOK_NAME"] = "qlora_finetune.ipynb"
wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

processor = AutoProcessor.from_pretrained(model_id)
# Right padding for training
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Highly optimized for speed/accuracy
    bnb_4bit_use_double_quant=True, # Saves extra memory at no speed cost
    bnb_4bit_compute_dtype=compute_dtype
)

# model = VoxtralForConditionalGeneration.from_pretrained(
#             model_id,
#             quantization_config=bnb_config,
#             attn_implementation="flash_attention_2",
#             device_map=device
#         )
# print(model)
# del model
# gc.collect()
# torch.cuda.empty_cache()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/vito/.netrc.
wandb: Currently logged in as: vitolus (vitolus-universit-ca-foscari-venezia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [2]:
def create_datasets(df, commands_list, test_size=0.1):
    train_cmds, eval_cmds = train_test_split(commands_list, test_size=test_size, random_state=42)

    def format_ds(frame):
        messages = []
        for _, row in frame.iterrows():
            path = os.path.join("data/synthesized_train_16k/", row["Audio_File"])
            if os.path.exists(path):
                messages.append([
                    {"role": "user", "content": [{"type": "audio", "path": path}]},
                    {"role": "assistant", "content": [{"type": "text", "text": row['Target_GLaDOS_Response']}]}
                ])
        return Dataset.from_dict({"messages": messages})

    df_train = df[df['User_Command'].isin(train_cmds)]
    df_eval = df[df['User_Command'].isin(eval_cmds)]
    print(f"Train rows: {len(df_train)} | Eval rows: {len(df_eval)}")
    return format_ds(df_train).shuffle(seed=42), format_ds(df_eval)

def get_prepared_model(model_id, quantization_config, device, compute_dtype, processor):
    model = VoxtralForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        attn_implementation="flash_attention_2",
        device_map=device,
        dtype=compute_dtype
    )
    # Prepare model for gradient training
    model = prepare_model_for_kbit_training(model)
    # Essential for preventing backward pass crashes with frozen encoders
    model.enable_input_require_grads()
    # # Revert the text embeddings back to bfloat16/float16 to match the Audio Encoder
    # model.get_input_embeddings().to(compute_dtype)
    # # Also ensure the output layer matches
    # if getattr(model, "get_output_embeddings", None) is not None:
    #     model.get_output_embeddings().to(compute_dtype)
    # # It is also good practice to ensure the audio encoder didn't get accidentally cast to float32
    # if hasattr(model, "audio_encoder"):
    #     model.audio_encoder.to(compute_dtype)
    model.config.update({
        "pad_token_id": processor.tokenizer.pad_token_id,
        "eos_token_id": processor.tokenizer.eos_token_id,
        "bos_token_id": processor.tokenizer.bos_token_id
    })
    return model

def make_voxtral_collate_fn(processor, compute_dtype):
    def collate_fn(batch):
        inputs_list = []
        labels_list = []
        for item in batch:
            # 1. Extract user message (contains audio) and assistant target text
            user_message = [item["messages"][0]]
            assistant_text = item["messages"][1]["content"][0]["text"]
            # 2. Tokenize ONLY the user prompt to bypass the validator
            prompt_inputs = processor.apply_chat_template(
                user_message,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt"
            )
            prompt_len = prompt_inputs["input_ids"].shape[1]
            # 3. Tokenize assistant text natively
            assistant_tokens = processor.tokenizer(
                assistant_text,
                add_special_tokens=False,
                return_tensors="pt"
            )
            # 4. Concatenate the Prompt + Assistant Text + EOS Token
            eos_tensor = torch.tensor([[processor.tokenizer.eos_token_id]])
            full_input_ids = torch.cat([
                prompt_inputs["input_ids"],
                assistant_tokens["input_ids"],
                eos_tensor
            ], dim=1)
            prompt_inputs["input_ids"] = full_input_ids
            if "attention_mask" in prompt_inputs:
                full_attention_mask = torch.cat([
                    prompt_inputs["attention_mask"],
                    assistant_tokens["attention_mask"],
                    torch.tensor([[1]])
                ], dim=1)
                prompt_inputs["attention_mask"] = full_attention_mask
            # 5. Create labels and mask the user prompt
            labels = full_input_ids.clone()
            labels[0, :prompt_len] = -100
            # Strip the arbitrary batch dim of 1 to prepare for manual stacking
            inputs_list.append({k: v[0] for k, v in prompt_inputs.items()})
            labels_list.append(labels[0])
        # 6. MANUALLY PAD BATCH (Replacing processor.pad)
        batch_padded = {}
        keys = inputs_list[0].keys()
        for key in keys:
            if key == "input_ids":
                batch_padded[key] = torch.nn.utils.rnn.pad_sequence(
                    [item[key] for item in inputs_list],
                    batch_first=True,
                    padding_value=processor.tokenizer.pad_token_id
                )
            elif key == "attention_mask":
                batch_padded[key] = torch.nn.utils.rnn.pad_sequence(
                    [item[key] for item in inputs_list],
                    batch_first=True,
                    padding_value=0
                )
            else:
                # Dynamically handle audio features (or any other extra keys)
                tensors = [item[key] for item in inputs_list]
                if isinstance(tensors[0], torch.Tensor):
                    try:
                        # If audio embeddings are exactly the same size, standard stack works
                        batch_padded[key] = torch.stack(tensors)
                    except RuntimeError:
                        # If audio files are different lengths, dynamically pad them with zeros
                        batch_padded[key] = torch.nn.utils.rnn.pad_sequence(
                            tensors,
                            batch_first=True,
                            padding_value=0.0
                        )
                else:
                    # Pass through non-tensor metadata (if mistral_common includes any)
                    batch_padded[key] = tensors
        # 7. Manually pad the labels with -100 to ignore empty space in loss calc
        batch_padded["labels"] = torch.nn.utils.rnn.pad_sequence(
            labels_list,
            batch_first=True,
            padding_value=-100
        )
        # 8. Safely cast floating point tensors (like the audio inputs) to your compute dtype
        for key, tensor in batch_padded.items():
            if isinstance(tensor, torch.Tensor) and torch.is_floating_point(tensor):
                batch_padded[key] = tensor.to(compute_dtype)
        return batch_padded
    return collate_fn

#### Dataset Formatting for Multimodal SFT

In [3]:
df = pd.read_csv("data/combined_multimodal_dataset_train.csv")
unique_commands = df['User_Command'].unique().tolist()
train_dataset, eval_dataset = create_datasets(df, unique_commands)
del df, unique_commands
gc.collect()

Train rows: 36120 | Eval rows: 4176


8

#### Hyperpatameters tuning

In [4]:
sweep_config = {
        'method': 'bayes', # Bayesian optimization (smarter than random search)
        'metric': {'name': 'eval/loss', 'goal': 'minimize'},
        'early_terminate': {
            'type': 'hyperband',
            'min_iter': 3, # Minimum number of iterations to run
            'eta': 1 # Aggressiveness of early stopping (higher = more aggressive). Hyperband will stop poorly performing runs early based on intermediate results, allowing more resources for promising configurations.
        },
        'parameters': {
            'learning_rate': {'distribution': 'log_uniform_values', 'min': 1e-6, 'max': 1e-3},
            'lora_r': {'values': [8, 16, 32]}, # Rank of the adapters
            'lora_alpha': {'values': [16, 32, 64]}, # Scaling factor
            'lora_dropout': {'values': [0.0, 0.05, 0.1]} # Dropout for regularization
        }
    }
sweep_id = wandb.sweep(sweep_config, project="Voxtral-GLaDOS-Multimodal")
base_model = get_prepared_model(model_id, bnb_config, device, compute_dtype, processor)

def sweep_train_step():
    with wandb.init() as run:
        config = wandb.config
        output_dir = f"models/voxtral-sweep-{run.id}"
        sweep_train_set = train_dataset.select(range(800))
        sweep_eval_set = eval_dataset.shuffle(42).select(range(200))

        # Dynamic LoRA Config from Sweep
        lora_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head", "multi_modal_projector.linear_1", "multi_modal_projector.linear_2"],
            use_rslora=True, # Empirically balances spectral weights across layers safely
            lora_dropout=config.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )
        model = get_peft_model(base_model, lora_config)

        # Dynamic Training Args
        training_args = SFTConfig(
            output_dir=output_dir,
            num_train_epochs=1,
           per_device_train_batch_size=2,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=8,
            max_grad_norm=1.0,
            eval_strategy="steps",
            eval_steps=10,
            save_strategy="no",
            load_best_model_at_end=False,
            gradient_checkpointing=True,
            dataloader_pin_memory=True,
            learning_rate=config.learning_rate,
            logging_steps=10,
            optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            remove_unused_columns=False,
            dataset_kwargs={"skip_prepare_dataset": True},
            report_to="wandb",
            loss_type="nll",
            use_liger_kernel=True,
            neftune_noise_alpha=5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.1,
            weight_decay=0.01,
            max_length=None
        )
        # Initialize Trainer
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=sweep_train_set,
            eval_dataset=sweep_eval_set,
            data_collator=make_voxtral_collate_fn(processor, compute_dtype),
            processing_class=processor
        )

        try:
            trainer.train()
        finally:
            model.unload()
            if 'trainer' in locals():
                trainer.optimizer = None
                trainer.lr_scheduler = None
                trainer.model = None
                trainer.train_dataset = None
                trainer.eval_dataset = None
                del trainer

            if 'model' in locals():
                del model

            gc.collect()
            gc.collect()
            torch.cuda.empty_cache()

print("Launching Weights & Biases Optimization Sweep...")
wandb.agent(sweep_id, function=sweep_train_step, count=5)

Create sweep with ID: ych3wbd9
Sweep URL: https://wandb.ai/vitolus-universit-ca-foscari-venezia/Voxtral-GLaDOS-Multimodal/sweeps/ych3wbd9


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Launching Weights & Biases Optimization Sweep...


...

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/transformers/trainer.py:1915: UserWarning: liger-kernel did not return token_accuracy when requested. The mean_token_accuracy metric will not be logged. This is unexpected; please report it to the liger-kernel repository.
  loss = self.compute_loss(model, inputs, num_items_in_batch=num_items_in_batch)


Step,Training Loss,Validation Loss,Num Tokens
10,2.467587,1.475850,65242.000000
20,1.576556,1.512309,130272.000000
30,1.582957,1.404185,195239.000000


/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/transformers/trainer.py:2982: UserWarning: liger-kernel did not return token_accuracy when requested. The mean_token_accuracy metric will not be logged. This is unexpected; please report it to the liger-kernel repository.
  loss, outputs = self.compute_loss(
Traceback (most recent call last):
  File "/tmp/ipykernel_4276/4246728047.py", line 79, in sweep_train_step
    trainer.train()
  File "/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/transformers/trainer.py", line 1433, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/transformers/trainer.py", line 1515, in _inner_training_loop
    self._run_epoch(
  File "/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/transformers/trainer.py", line 1743, in _run_epoch
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^

eval/loss,▆█▁
eval/num_tokens,▁▅█
eval/runtime,▁▂█
eval/samples_per_second,█▇▁
eval/steps_per_second,█▇▁
train/epoch,▁▁▄▄██
train/global_step,▁▁▅▅██
train/grad_norm,▅█▁
train/learning_rate,█▅▁
train/loss,█▁▁
+1,...


Traceback (most recent call last):
  File "/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/wandb/agents/pyagent.py", line 314, in _run_job
    self._function()
  File "/tmp/ipykernel_4276/4246728047.py", line 79, in sweep_train_step
    trainer.train()
  File "/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/transformers/trainer.py", line 1433, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/transformers/trainer.py", line 1515, in _inner_training_loop
    self._run_epoch(
  File "/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/transformers/trainer.py", line 1743, in _run_epoch
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/trl/trainer/sft_trainer.py", line 1826, in training_step
  

In [5]:
api = wandb.Api()
wandb_sweep = api.sweep(f"{api.default_entity}/Voxtral-GLaDOS-Multimodal/{sweep_id}")
best_params = wandb_sweep.best_run().config
del base_model, sweep_id, sweep_config
gc.collect()
torch.cuda.empty_cache()
with open("models/best_sweep_params.json", "w") as f:
    json.dump(best_params, f, indent=4)
print(f"Best Sweep Parameters: {best_params}")

wandb: Sorting runs by +summary_metrics.eval/loss


Best Sweep Parameters: {'bf16': True, 'fp16': False, 'fsdp': None, 'seed': 42, 'tf32': None, 'debug': [], 'dtype': 'bfloat16', 'optim': 'paged_adamw_8bit', 'lora_r': 32, 'do_eval': True, 'packing': False, 'project': 'huggingface', 'use_cpu': False, 'do_train': False, 'id2label': {'0': 'LABEL_0', '1': 'LABEL_1'}, 'label2id': {'LABEL_0': 0, 'LABEL_1': 1}, 'run_name': None, 'data_seed': None, 'deepspeed': None, 'eos_token': '<EOS_TOKEN>', 'hub_token': '<HUB_TOKEN>', 'log_level': 'passive', 'loss_type': 'nll', 'max_steps': -1, 'pad_token': '<PAD_TOKEN>', 'report_to': ['wandb'], 'use_cache': False, 'adam_beta1': 0.9, 'adam_beta2': 0.999, 'do_predict': False, 'eval_delay': 0, 'eval_steps': 10, 'local_rank': -1, 'lora_alpha': 16, 'max_length': None, 'model_type': 'voxtral', 'optim_args': None, 'output_dir': 'models/voxtral-sweep-ao0sea61', 'save_steps': 500, 'vocab_size': 131072, 'ddp_backend': None, 'ddp_timeout': 1800, 'fsdp_config': None, 'hidden_size': 3072, 'label_names': None, 'logging_

#### Supervised Fine-Tuning with TRL's SFTTrainer

In [ ]:
best_params = {
    'learning_rate': 3e-4,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.1
    }
if os.path.exists("models/best_sweep_params.json"):
    with open("models/best_sweep_params.json", "r") as f:
        best_params = json.load(f)
output_dir = "models/voxtral-glados-sft"
last_checkpoint = get_last_checkpoint(output_dir) if os.path.exists(output_dir) else None

print(f"Loading {model_id} for final production run...")
model = get_prepared_model(model_id, bnb_config, device, compute_dtype, processor)

lora_config = LoraConfig(
    r=best_params['lora_r'],
    lora_alpha=best_params['lora_alpha'],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head", "multi_modal_projector.linear_1", "multi_modal_projector.linear_2"],
    use_rslora=True, # Empirically balances spectral weights across layers safely
    lora_dropout=best_params['lora_dropout'],
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    max_grad_norm=1.0,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=False,
    gradient_checkpointing=True,
    # gradient_checkpointing_kwargs={"use_reentrant": False},
    # dataloader_num_workers=4,
    # dataloader_prefetch_factor=2,
    dataloader_pin_memory=True,
    learning_rate=best_params['learning_rate'],
    logging_steps=10,
    num_train_epochs=1,
    optim="paged_adamw_8bit", # paged_adamw_8bit use ram if vram is saturated (paging)
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False, # Crucial so the collator receives the dicts
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to="wandb",
    loss_type="nll",
    use_liger_kernel=True,
    neftune_noise_alpha=5, # Add a small amount of noise to the activations during training to improve generalization
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_length=None
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=make_voxtral_collate_fn(processor, compute_dtype),
    processing_class=processor,
    peft_config=lora_config
)
trainer.model.print_trainable_parameters()

In [ ]:
try:
    if last_checkpoint:
        print(f"Resuming training from {last_checkpoint}...")
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print("Starting a new training run...")
        trainer.train()
    # Save the final adapter weights
    trainer.save_model(os.path.join(output_dir, "final_adapters"))
    print("Training complete. Adapters saved.")
finally:
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    wandb.finish()